In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")
from langchain.chat_models import init_chat_model
model = init_chat_model(
    "groq:openai/gpt-oss-120b"
)

In [3]:
response = model.invoke("Hello, how are you?")
response

AIMessage(content="Hello! I'm doing great, thank you for asking. How can I assist you today?", additional_kwargs={'reasoning_content': 'We need to respond as ChatGPT. The system message says we are ChatGPT, we must follow policies. The user says "Hello, how are you?" This is a casual greeting. We respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 77, 'total_tokens': 147, 'completion_time': 0.14475746, 'completion_tokens_details': {'reasoning_tokens': 43}, 'prompt_time': 0.002967416, 'prompt_tokens_details': None, 'queue_time': 0.332984707, 'total_time': 0.147724876}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f1b2c-d77d-77f0-b6b5-06179aff9440-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 70, 'total_tokens': 147, 'output_token_details': {'reasoning

In [4]:
from langchain.tools import tool
@tool
def get_weather(location: str) -> str:
    """Get the current weather in a given location"""
    return f"The current weather in {location} is sunny."

model_with_tools = model.bind_tools([get_weather])

In [7]:
response = model_with_tools.invoke("What is the weather in New York?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
   
    print(f"args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'The user asks: "What is the weather in New York?" We need to fetch current weather. Use function get_weather with location "New York".', 'tool_calls': [{'id': 'fc_53f63a97-17ec-4bd9-8fe3-65acb5fdb27d', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 130, 'total_tokens': 189, 'completion_time': 0.122678295, 'completion_tokens_details': {'reasoning_tokens': 31}, 'prompt_time': 0.005145559, 'prompt_tokens_details': None, 'queue_time': 0.333265263, 'total_time': 0.127823854}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f1b3c-42ed-7000-af4e-9fc3bfbb180c-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'fc_53f63a97-17ec-4bd9-8fe3-65acb5f

## Tool Execution Loop

In [ ]:
# step 1: Model generate tool call
messages = [{"role":"user","content":"What is the weather in Lahore?"}]
ai_msg = model_with_tools.invoke(messages)

messages.append(ai_msg)

#Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather(tool_call)